

## Setup

In [ ]:
import os as _os
import subprocess as _subprocess

def is_colab_env() -> bool:
  """
  check if we are in Google Colab Hosted Runtime Environment
  """
  try:
    import google.colab
    return True and _os.path.exists("/content")
  except ImportError:
    return False


def get_git_root():
  try:
    return _subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True ).strip()
  except _subprocess.CalledProcessError:
    return None


def colab_repo_sync(repo_url = None, repo_subdir = "."):
  repo_path = get_git_root()
  if repo_path is None:
    if is_colab_env() and repo_url is not None:
      repo_path = "cloned_repo"
      _subprocess.call(['git', 'clone', f'{repo_url}', f'{repo_path}']) # !git clone {repo_url} {repo_path}
      print("Cloning complete.")

      _os.chdir(repo_path) # change directory to cloned repo worktree
  else:
    _os.chdir(repo_path) # change directory to cloned repo worktree

  _os.chdir(repo_subdir) # change directory to the one where this notebook is located in the cloned repo
      

colab_repo_sync(
  repo_url="https://github.com/somombo/sort-bench.git",
  repo_subdir="lab",
)

if is_colab_env(): # check if we are in Google Colab Hosted Runtime Environment
    from env_setup import setup_lean, setup_rust
    setup_lean("v4.24.0") # Then we install the Lean toolchain ~~snd install the Rust (cargo) toolchain~~

    from google.colab import output
    output.enable_custom_widget_manager()
    print("Enabled colab custom widget manager")


print("$PWD:", _subprocess.check_output(['pwd'], text=True))
# _subprocess.call(['git', 'clone', f'{repo_url}', f'{repo_path}'], text=True)

## Experiment Config

In [ ]:
from impalab_py import Impa
import json

impa = Impa(
  root_dir="..",
)


Let us run a warmup round of the benchmark to pre-build build all the executables

In [ ]:
_build_result = impa.build(
  components_dir="./components",
)
assert _build_result, "Initial build failed"

In [ ]:
_test_results = impa.run(
  generator={
    "name": "somombo_unifshuffle_multi",
    "seed": 42,
    "args": [json.dumps([
      {
        "cardinality": 19,
        "multiplicity": 13,
        "swaps": 2,
        "descending": True,
        "runs": 7,
      }, 
      {
        "cardinality": 17,
        "multiplicity": 11,
        "swaps": None,
        "descending": False,
        "runs": 5,
      }
    ])],
  }, 
  reps=3, 
  tasks=[
    {'executor': 'lean', 'args': ['Somombo.qs.bentleyMcIlroy.classic']},
    {'executor': 'cpp', 'args': ['std::sort']},
  ],
  attributes={
    "experiment_name": "Warmup-Experiment",
  },
)

In [ ]:
from exploration import ExplorerFromLabDf

ExplorerFromLabDf(_test_results).get_clean_data().tail()

In [ ]:
from sample_factors import sample_spaced, get_factors, get_hcns